# Olist Feature Advisor Example

This notebook uses the package notebook utilities to resolve the local `featurizer_config.yaml` workspace, then reads Olist metadata and runs the feature advisor. The advisor is executed in rule-based mode with a graph modeling intent.

In [1]:
import os
from pathlib import Path

import pandas as pd
from featurization.notebook_utils import build_notebook_resolver, get_featurization_artifact_paths
from featurization.feature_advisor_util import FeatureAdvisorUtil, FeatureAdvisorPromptConfig

# 1) Resolve the notebook workspace, load the current config, and build a resolver
cwd = Path.cwd()
if cwd.name == "notebooks" and (cwd.parent / "featurizer_config.yaml").exists():
    notebook_dir = cwd
elif (cwd / "notebooks").is_dir() and (cwd / "featurizer_config.yaml").exists():
    notebook_dir = cwd / "notebooks"
else:
    notebook_dir = cwd

resolver = build_notebook_resolver(str(notebook_dir))
workspace_root = Path(resolver.working_dir)
config = resolver.config

print("Notebook dir:", notebook_dir)
print("Workspace root:", workspace_root)
print("Loaded config keys:", list(config.keys()))

# 2) Resolve metadata and current featurization input paths from the workspace resolver
metadata_path = workspace_root / config["metadata_file"]
input_data_path = workspace_root / config.get("featurization_input_data", config.get("merged_raw_file"))
if not input_data_path.exists():
    fallback_path = workspace_root / config.get("dd_cleaner_output_dir", "data/dd_cleaner") / config.get("clean_output_filename", "")
    if fallback_path.exists():
        input_data_path = fallback_path
        print("Using cleaned fallback input data:", input_data_path)
    else:
        raise FileNotFoundError(
            f"Input data file not found at {input_data_path}. "
            f"Expected merged raw file at {workspace_root / config.get("merged_raw_file", "")} or cleaned dataset at {fallback_path}."
        )

print("Metadata path:", metadata_path)
print("Input data path:", input_data_path)

artifact_paths = get_featurization_artifact_paths(resolver)
print("Featurization artifact paths:")
for name, path in artifact_paths.items():
    print(f"- {name}: {path}")

# 3) Load metadata and build the advisor
if not metadata_path.exists():
    raise FileNotFoundError(
        f"Metadata file not found at {metadata_path}. "
        "Update `metadata_file` in featurizer_config.yaml or place the file there."
    )

metadata = pd.read_csv(metadata_path)
print("Metadata rows:", len(metadata))
print(metadata.head(3))

prompt_config = FeatureAdvisorPromptConfig.load_from_package()
advisor = FeatureAdvisorUtil(resolver=resolver, prompt_config=prompt_config)

print("Advisor output directory:", advisor.feature_advisor_dir)
print("Recommendation CSV path:", advisor.recommendations_csv_path)
print("Recommendation MD path:", advisor.recommendations_md_path)

# 4) Generate rule-based recommendations using the current merged input dataset
if not input_data_path.exists():
    raise FileNotFoundError(
        f"Input data file not found at {input_data_path}. "
        "Run the featurization pipeline or update `merged_raw_file` / `featurization_input_data` in featurizer_config.yaml."
    )

input_data = pd.read_csv(input_data_path)
rows, columns = input_data.shape
print(f"Input dataset shape: {rows} rows x {columns} columns")

dataset_type = resolver.structural_type
print("Dataset structural type:", dataset_type)

recommendations = advisor.recommend(
    metadata=metadata,
    model_intent="graph",
    input_data=input_data,
    use_rules=True,
)

print("Recommendations generated by rule-based advisor")
print(recommendations.head(20))

recommendations


Notebook dir: /home/rajiv/programming/kmds_migration/olist_migration/notebooks
Workspace root: /home/rajiv/programming/kmds_migration/olist_migration
Loaded config keys: ['working_dir', 'structural_type', 'country_code', 'featurization_input_data', 'metadata_file', 'featurization_output_dir', 'sp_freq_prod_file', 'sp_freq_prod_parquet', 'pipeline']
Metadata path: /home/rajiv/programming/kmds_migration/olist_migration/data_dictionary/olist_example_dd.csv
Input data path: /home/rajiv/programming/kmds_migration/olist_migration/data/dd_cleaner/olist_daily_orders_prepared_clean.csv
Featurization artifact paths:
- featurized_dataset_path: /home/rajiv/programming/kmds_migration/olist_migration/data/featurized_data.csv
- model_ready_dataset_path: /home/rajiv/programming/kmds_migration/olist_migration/data/model_ready_numeric_data.csv
- feature_selection_knee_curve_path: None
Metadata rows: 9013
                          attribute  \
0                               woy   
1  00088930e925c41fd95

,attribute,recommended_method,rationale
0,dataset,Feature selection recommended,This dataset is wide and short. Prioritize fea...
